# Notebook 2b: Extract the PlantVillage training features

**Why we need this.** Notebook 02 saved the model's features for the *validation* and *test* images, but not for the
**training** images. Several of our methods need the training features:

* **Moment matching** (the paper's version) re-shapes the training features and trains a new classifier on them.
* **DANN** and **JDOT** also learn from the source training features.

So we run the training images through the trained network once, and save the 2048 numbers per image.

**What it does:** loads the trained ResNet-50 from notebook 02, passes every PlantVillage training image through it
(no learning, only reading), and saves the features. It needs PyTorch and ideally a GPU (about 18,000 images, a few
minutes on a GPU).

**Quick test first:** set `LIMIT = 256` to process only 256 images. If that works, set `LIMIT = None` for the real run.

### If it is slow
Reading 18,000 images through ResNet-50 should take only a few minutes **on a GPU**. When it takes much longer, one of
three things is the cause, and the progress line printed during extraction tells you which:

| Symptom | Cause | Fix |
|---|---|---|
| Step 1 says `Using device: cpu` | no GPU is being used (ResNet-50 on a CPU is 20 to 50 times slower) | use a GPU (Colab or Kaggle offer free ones), or use `MAX_PER_CLASS` below |
| "waiting for images" is **high** (above ~30%) | the GPU is idle while images are loaded and decoded | more `NUM_WORKERS`, and copy the images to a **local** disk (not Google Drive or a network folder) |
| "waiting for images" is low, speed still low | the model itself is the limit | raise `BATCH_SIZE`, or set `USE_FP16 = True` |

## Step 1: Imports and settings

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

In [12]:
# ---- Edit these -------------------------------------------------------------
LIMIT = None                  # None = all training images (the real run). A small number = quick test.
MAX_PER_CLASS = None              # optional speed-up: use at most this many images per class (e.g. 300). None = all.

SOURCE_METHOD = "source_only"     # whose trained model we use
TRAIN_SEED = 42

MANIFEST = Path("outputs/master_manifest.csv")
PV_ROOT = Path(r"F:\mphil\pv") 
OUT_DIR = Path("outputs")

IMAGE_SIZE = 224
BATCH_SIZE = 128                  # bigger batches keep a GPU busier (lower it if you run out of GPU memory)
NUM_WORKERS = 0                   # processes that load images in parallel. Try 2 to 8 (about your CPU cores). 0 = slow.
USE_FP16 = False                  # True = half-precision on the GPU: faster, but numbers differ very slightly
# -----------------------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device, "| LIMIT =", LIMIT, "| MAX_PER_CLASS =", MAX_PER_CLASS)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("*** NO GPU FOUND. ResNet-50 on a CPU will take a very long time for 18,000 images. ***")
    print("*** Use a GPU, or set MAX_PER_CLASS (e.g. 300) to use fewer images.                ***")
if LIMIT is not None:
    print("*** QUICK TEST: only", LIMIT, "images. Set LIMIT = None for the real run. ***")

Using device: cuda | LIMIT = None | MAX_PER_CLASS = None
GPU: NVIDIA GeForce RTX 2080


## Step 2: The training images and their labels

We take the images with role `pv_train` from the manifest. The class-to-number mapping must be **identical** to
notebook 02's, so we read the class list that notebook 02 saved instead of rebuilding it.

In [13]:
saved = np.load(OUT_DIR / f"outputs_{SOURCE_METHOD}_seed{TRAIN_SEED}.npz", allow_pickle=True)
classes = list(saved["classes"])
class_to_id = {name: i for i, name in enumerate(classes)}

manifest = pd.read_csv(MANIFEST, keep_default_na=False)
train = manifest[manifest["role"] == "pv_train"].reset_index(drop=True)
train["label"] = train["class"].map(class_to_id)
assert train["label"].notna().all(), "Some training classes are not in the saved class list"
train["label"] = train["label"].astype(int)
train["full_path"] = [str(PV_ROOT / p) for p in train["path"]]

if MAX_PER_CLASS is not None:
    # Shuffle, then keep the first MAX_PER_CLASS images of every class (so every class stays represented)
    train = train.sample(frac=1, random_state=0).groupby("label").head(MAX_PER_CLASS).reset_index(drop=True)
    print(f"Using at most {MAX_PER_CLASS} images per class")

if LIMIT is not None:
    train = train.sample(LIMIT, random_state=0).reset_index(drop=True)

assert Path(train["full_path"].iloc[0]).exists(), "Image not found: check PV_ROOT"
print(len(train), "training images,", len(classes), "classes")

18601 training images, 21 classes


## Step 3: Load the trained model

We rebuild the same ResNet-50 and load the best weights saved by notebook 02. As a safety check, if notebook 02 also
saved the final layer inside its outputs file, we confirm that the two copies are identical.

In [14]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class ImageTable(Dataset):
    def __init__(self, table):
        self.paths = table["full_path"].tolist()
        self.labels = table["label"].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return transform(Image.open(self.paths[i]).convert("RGB")), self.labels[i]


loader = DataLoader(ImageTable(train), batch_size=BATCH_SIZE, shuffle=False,      # shuffle=False keeps the order
                    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
                    persistent_workers=(NUM_WORKERS > 0))                              # keep workers alive between batches

In [15]:
# Pick the real (non-debug) checkpoint from notebook 02
candidates = [p for p in sorted((OUT_DIR / "checkpoints").glob(f"{SOURCE_METHOD}_seed{TRAIN_SEED}*.pt"))
              if "DEBUG" not in p.name]
print("Checkpoint files found:", [p.name for p in candidates])
assert candidates, "No trained model found in outputs/checkpoints. Copy the .pt file from where you trained."

model = models.resnet50(weights=None)                         # no download: we load our own trained weights
model.fc = nn.Linear(model.fc.in_features, len(classes))
model.load_state_dict(torch.load(candidates[0], map_location=device))
model = model.to(device).eval()

body = nn.Sequential(*list(model.children())[:-1])            # the network without its final layer

if "fc_weight" in saved.files:
    same = np.allclose(model.fc.weight.detach().cpu().numpy(), saved["fc_weight"], atol=1e-6)
    print("Final layer matches the one saved by notebook 02:", same)
    assert same, "The checkpoint and the saved outputs come from different training runs"

Checkpoint files found: ['source_only_seed42.pt']
Final layer matches the one saved by notebook 02: True


## Step 4: Extract the features

Same as notebook 02's `predict`: for each batch we keep the 2048-number features. Nothing is learned.

We time the loop and print two numbers every 20 batches: **images per second**, and the **share of time spent waiting
for images to be loaded**. They tell you where any slowness comes from (see the table at the top).

In [16]:
torch.backends.cudnn.benchmark = True        # lets the GPU pick its fastest convolution method (our input size is fixed)

features, labels = [], []
start_time = time.time()
wait_time = 0.0                              # time spent waiting for the next batch of images
batch_end = time.time()

with torch.no_grad():
    for i, (images, y) in enumerate(loader):
        wait_time += time.time() - batch_end             # how long we waited for this batch to arrive
        images = images.to(device, non_blocking=True)

        if USE_FP16 and device.type == "cuda":
            with torch.autocast("cuda", dtype=torch.float16):
                out = body(images)
        else:
            out = body(images)

        features.append(out.flatten(1).float().cpu().numpy())   # .cpu() waits for the GPU, so timings are honest
        labels.append(y.numpy())
        batch_end = time.time()

        if i % 20 == 0:
            elapsed = batch_end - start_time
            done = min((i + 1) * BATCH_SIZE, len(train))
            remaining = (len(train) - done) / (done / elapsed)
            print(f"batch {i + 1} of {len(loader)} | {done / elapsed:.0f} images/s | "
                  f"waiting for images {100 * wait_time / elapsed:.0f}% of the time | about {remaining / 60:.1f} min left")

features = np.concatenate(features)
labels = np.concatenate(labels)
print(f"Done in {(time.time() - start_time) / 60:.1f} min. features: {features.shape}, labels: {labels.shape}")

batch 1 of 146 | 76 images/s | waiting for images 64% of the time | about 4.1 min left
batch 21 of 146 | 104 images/s | waiting for images 80% of the time | about 2.6 min left
batch 41 of 146 | 105 images/s | waiting for images 80% of the time | about 2.1 min left
batch 61 of 146 | 104 images/s | waiting for images 81% of the time | about 1.7 min left
batch 81 of 146 | 104 images/s | waiting for images 81% of the time | about 1.3 min left
batch 101 of 146 | 104 images/s | waiting for images 81% of the time | about 0.9 min left
batch 121 of 146 | 104 images/s | waiting for images 81% of the time | about 0.5 min left
batch 141 of 146 | 104 images/s | waiting for images 81% of the time | about 0.1 min left
Done in 3.0 min. features: (18601, 2048), labels: (18601,)


### Sanity check

If everything is right, the trained final layer should classify its own *training* images almost perfectly.
Anything much lower means the wrong checkpoint was loaded.

In [17]:
with torch.no_grad():
    scores = model.fc(torch.from_numpy(features).to(device)).cpu().numpy()

train_accuracy = float((scores.argmax(axis=1) == labels).mean())
print(f"Accuracy of the trained model on these training images: {train_accuracy:.4f}")
assert train_accuracy > 0.95, "Suspiciously low: wrong checkpoint, wrong class order, or wrong PV_ROOT?"

Accuracy of the trained model on these training images: 0.9999


## Step 5: Save

In [18]:
suffix = f"_LIMIT{LIMIT}" if LIMIT is not None else ""
output_file = OUT_DIR / f"features_pv_train_{SOURCE_METHOD}_seed{TRAIN_SEED}{suffix}.npz"

np.savez_compressed(output_file,
                    features=features.astype(np.float16),      # half precision saves disk space
                    labels=labels,
                    paths=train["path"].to_numpy())
print("Saved", output_file, f"({output_file.stat().st_size / 1e6:.1f} MB)")

Saved outputs\features_pv_train_source_only_seed42.npz (68.9 MB)


### Next
If this was the quick test (`LIMIT = 256`), go back to Step 1, set `LIMIT = None`, restart the kernel, and run again.
The real file has **no** `_LIMIT` in its name, and that's the one notebook 4a reads.